# 🩺 Predicción de Riesgo de Enfermedad Cardiovascular / Diabetes
**Autor:** Jorge Sánchez Hernández  
**Proyecto:** Machine Learning supervisado  
**Datos:** Encuesta de salud España 2025 *(Ministerio de Sanidad)*

---
**Índice**
1. Introducción y objetivo
2. Carga de datos
3. Análisis exploratorio (EDA)
4. Limpieza de datos
5. Preprocesamiento y división train/test
6. Modelo base: Regresión Logística
7. Random Forest + GridSearchCV
8. XGBoost + RandomizedSearchCV
9. Comparación de modelos
10. Evaluación final en test
11. Importancia de variables
12. Conclusiones

## 1. Introducción y objetivo

- ¿Qué problema de salud pública abordas?
- ¿Qué variable quieres predecir?
- ¿Por qué es importante minimizar los falsos negativos (priorizar **Recall**)?

In [1]:
# Librerías utilizadas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

# Que boost utilizamos?

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'seaborn'

## 2. Carga de datos


In [ ]:
df_adulto = pd.read_csv("../data/ESdEadulto_2025.tab", sep="\t", low_memory=False)
df_hogar = pd.read_csv("../data/ESdEhogar_2025.tab", sep="\t", low_memory=False)

print("Adulto:", df_adulto.shape)
print("Hogar:", df_hogar.shape)
df_adulto.head()

NameError: name 'pd' is not defined

## 3. Análisis exploratorio (EDA)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Valores nulos (nº y %)
nulos = pd.DataFrame({"nulos": df.isnull().sum(),
                      "porcentaje": (df.isnull().mean() * 100).round(2)})
nulos[nulos["nulos"] > 0].sort_values("porcentaje", ascending=False)

In [ ]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())

### 3.1 Variable objetivo
*Define qué columna(s) usas. Si combinas enfermedad cardiovascular y diabetes en una sola variable, explica el criterio.*

In [ ]:
# TODO: ajusta los nombres de columna de tu dataset
# Opción A: ya existe una columna binaria
# df["target"] = df["NOMBRE_COLUMNA"]

# Opción B: combinar varias columnas (1 = tiene alguna de las dos)
# df["target"] = ((df["diabetes"] == 1) | (df["cardiovascular"] == 1)).astype(int)

# print(df["target"].value_counts())
# print(df["target"].value_counts(normalize=True).round(3))

In [ ]:
# Distribución de la variable objetivo (descomenta cuando exista 'target')
# sns.countplot(x="target", data=df)
# plt.title("Distribución de la variable objetivo")
# plt.show()

### 3.2 Distribución de variables y relación con el objetivo

In [ ]:
# Variables numéricas: histogramas
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols].hist(figsize=(14, 10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas: nº de categorías
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(col, "->", df[col].nunique(), "categorías")

In [ ]:
# Ejemplo: relación de una variable con el objetivo (cambia 'edad' por la tuya)
# sns.boxplot(x="target", y="edad", data=df)
# plt.show()

# Ejemplo categórica:
# sns.countplot(x="tabaco", hue="target", data=df)
# plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Matriz de correlación")
plt.show()

**Observaciones del EDA:** *(escribe aquí lo que has visto: desbalanceo, variables con más correlación, outliers, etc.)*

## 4. Limpieza de datos

In [ ]:
# Eliminar duplicados
df = df.drop_duplicates()

# Eliminar columnas que no sirven (identificadores, etc.)
# df = df.drop(columns=["id"])

# Nulos: ejemplos sencillos
# Numéricas -> mediana
# df["imc"] = df["imc"].fillna(df["imc"].median())
# Categóricas -> moda
# df["tabaco"] = df["tabaco"].fillna(df["tabaco"].mode()[0])
# O eliminar filas con nulos:
# df = df.dropna()

print(df.shape)
print("Nulos restantes:", df.isnull().sum().sum())

*Justifica aquí las decisiones de limpieza (por qué mediana, por qué eliminas filas, etc.).*

## 5. Preprocesamiento y división train/test

In [ ]:
# Separar X e y
X = df.drop(columns=["target"])
y = df["target"]

# Codificar variables categóricas
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

In [ ]:
# División train/test (estratificada para mantener la proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# Escalado (solo para Regresión Logística). Se ajusta con train y se aplica a test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

**Desbalanceo de clases:** si la clase positiva es minoritaria, usamos `class_weight="balanced"` en los modelos (y `scale_pos_weight` en XGBoost) para que el modelo no ignore a los enfermos.

In [ ]:
# Peso para XGBoost = nº negativos / nº positivos
peso_pos = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(peso_pos, 2))

## 6. Modelo base: Regresión Logística

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)

# Validación cruzada (priorizamos recall)
cv_recall = cross_val_score(log_reg, X_train_sc, y_train, cv=5, scoring="recall")
cv_f1 = cross_val_score(log_reg, X_train_sc, y_train, cv=5, scoring="f1")

print("Recall CV:", cv_recall.round(3), "-> media", cv_recall.mean().round(3))
print("F1 CV:    ", cv_f1.round(3), "-> media", cv_f1.mean().round(3))

In [ ]:
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

print(classification_report(y_test, y_pred_lr))

**Comentario:** *(¿qué tal funciona el modelo base? ¿Recall alto pero precisión baja?)*

## 7. Random Forest + GridSearchCV

In [ ]:
rf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5, 10]
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring="recall", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Mejores parámetros:", grid_rf.best_params_)
print("Mejor recall CV:", round(grid_rf.best_score_, 3))

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

## 8. XGBoost + RandomizedSearchCV

In [ ]:
xgb = XGBClassifier(scale_pos_weight=peso_pos, eval_metric="logloss", random_state=RANDOM_STATE)

param_dist_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0]
}

rand_xgb = RandomizedSearchCV(xgb, param_dist_xgb, n_iter=10, cv=5, scoring="recall",
                              random_state=RANDOM_STATE, n_jobs=-1)
rand_xgb.fit(X_train, y_train)

print("Mejores parámetros:", rand_xgb.best_params_)
print("Mejor recall CV:", round(rand_xgb.best_score_, 3))

In [ ]:
best_xgb = rand_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

## 9. Comparación de modelos

In [ ]:
modelos = {
    "Regresión Logística": (y_pred_lr, log_reg.predict_proba(X_test_sc)[:, 1]),
    "Random Forest": (y_pred_rf, best_rf.predict_proba(X_test)[:, 1]),
    "XGBoost": (y_pred_xgb, best_xgb.predict_proba(X_test)[:, 1]),
}

resultados = []
for nombre, (pred, proba) in modelos.items():
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, proba),
    })

tabla = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabla

In [ ]:
tabla[["Recall", "F1", "ROC AUC"]].plot(kind="bar", figsize=(9, 5))
plt.title("Comparación de modelos")
plt.ylabel("Métrica")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

## 10. Evaluación final en test
*Elige el mejor modelo según Recall y F1 y justifica por qué.*

In [ ]:
# Cambia por el modelo ganador
mejor_modelo = best_xgb
y_pred_final = y_pred_xgb

cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(cm, display_labels=["Sin enfermedad", "Con enfermedad"]).plot(cmap="Blues")
plt.title("Matriz de confusión - modelo final")
plt.show()

**Interpretación:** *(cuántos falsos negativos hay, qué implicaría en la práctica, etc.)*

## 11. Importancia de variables

In [ ]:
importancias = pd.Series(mejor_modelo.feature_importances_, index=X_train.columns)
top = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=top.values, y=top.index)
plt.title("Top 15 variables más importantes")
plt.xlabel("Importancia")
plt.show()

**Interpretación desde salud pública:** *(qué factores de riesgo destacan: edad, IMC, tabaco, actividad física... ¿tiene sentido clínico?)*

## 12. Conclusiones
- **Resultados:** *(mejor modelo y métricas)*
- **Factores de riesgo más relevantes:** *...*
- **Limitaciones:** *(datos autodeclarados, desbalanceo, variables que faltan...)*
- **Próximos pasos:** *(más modelos, ajustar umbral de decisión, más datos...)*